# M01 – Modules and the Standard Library

**PCAP Alignment**: Section 1 (1.1–1.4) – Import and use modules; math, random, platform  
**Professional Focus**: Multi-file programs, standard library, discoverability.

---

## Learning Outcomes

- Import modules using all common variants (import, from import, import as, import *).

- Use `dir()` and `sys.path` to discover and control module resolution.

- Use the **math**, **random**, and **platform** modules for real tasks.

---

## Table of Contents

1. Import Variants (PCAP 1.1)
2. dir() and sys.path (PCAP 1.1)
3. The math Module (PCAP 1.2)
4. The random Module (PCAP 1.3)
5. The platform Module (PCAP 1.4)
6. Built-in and Related Functions
7. Common Errors and Edge Cases
8. More Examples
9. Practice

In [ ]:
import math, random, platform
print(math.sqrt(16), random.choice([1,2,3]), platform.system())

---
## 1. Import Variants (PCAP 1.1)

Python offers several ways to bring module code into your program. Each form has trade-offs:

- **import math** — Imports the whole module. You then call **math.sqrt(4)**. The module name is the **namespace**, so it's always clear where a name comes from. Best when you use many names from the module.

- **from math import sqrt** — Imports only **sqrt** and puts it in the current scope. You call **sqrt(4)** directly. Shorter and readable when you need just a few names.

- **from math import sqrt as sq** — Same as above but renames **sqrt** to **sq**. Use when the name would clash with a local variable or another import (e.g. **sq** to avoid shadowing a variable named **sqrt**).

- **from math import *** — Imports every public name from **math** into the current scope. **Avoid in production**: it's unclear where names come from, and new names added to the module can break your code.

- **Nested packages**: Use **from pkg.subpkg import mod** to import a module inside a package; then use **mod.something** or **from pkg.subpkg.mod import foo**.

In [ ]:
import math
from math import floor, ceil
from math import sqrt as sq
print(math.sqrt(16), floor(3.7), ceil(3.2), sq(9))

In [ ]:
# Example: from math import * — AVOID in production (namespace pollution)
# All math names (sqrt, sin, cos, pi, ...) get dumped into current scope
from math import *
# Now sqrt, pi, sin, etc. are available directly — but unclear where they came from
print("sqrt(25) =", sqrt(25), "| pi =", pi)
# If your code had a variable named 'sqrt', it would be overwritten — hard to debug!

In [ ]:
# Mixing styles in one program: qualified vs direct names
import math
from math import pi
# math.pi and pi refer to the same value
print("math.pi == pi:", math.pi == pi)
print("Using qualified name:", math.pi)
print("Using imported name:", pi)

---
## 2. dir() and sys.path (PCAP 1.1)

**Discovering what a module provides**

- **dir()** with no arguments returns a list of names in the **current scope** (e.g. in the notebook or module). **dir(math)** returns the list of attribute names of the **math** module — functions, constants like **pi**, and other objects. Use it to explore a module before reading the docs.

- **sys.path** is a list of directory path strings. When you write **import foo**, Python searches these directories in order and uses the first place where **foo** is found (as a **foo.py** file or a **foo** package directory). The first entry is often the directory of the script you ran, or the current working directory. You can **append** or **insert** paths to add your own package roots (e.g. **sys.path.append(".")** or a project lib folder).

In [ ]:
import sys
print("dir(math)[:6] =", dir(math)[:6])
print("sys.path[0] =", sys.path[0])

In [ ]:
# Example: sys.path — see where Python looks for modules; append custom paths
import sys
print("First 3 entries in sys.path:")
for i, p in enumerate(sys.path[:3]):
    print(f"  [{i}] {p}")
# Appending a path: sys.path.append(".") or sys.path.insert(0, "/my/custom/lib")

In [ ]:
# Inspect how many names math exports and show a few more
math_names = [n for n in dir(math) if not n.startswith("_")]
print("Number of public names in math:", len(math_names))
print("Sample:", math_names[:10])

---
## 3. The math Module (PCAP 1.2)

### Concepts

The **math** module provides mathematical functions. All angles in **math** trigonometric functions are in **radians** (not degrees). You must **import math** (or from math import ...) before use.

### Function Reference (PCAP 1.2)

| Function | Description | Example |
|----------|-------------|--------|
| **ceil(x)** | Smallest integer >= x | ceil(3.2) -> 4 |
| **floor(x)** | Largest integer <= x | floor(3.8) -> 3 |
| **trunc(x)** | Truncate toward zero (same as int for float) | trunc(-3.7) -> -3 |
| **factorial(n)** | n! (n must be non-negative int) | factorial(5) -> 120 |
| **hypot(x, y)** | sqrt(x*x + y*y); Euclidean distance from (0,0) | hypot(3, 4) -> 5.0 |
| **sqrt(x)** | Square root; x must be >= 0 | sqrt(16) -> 4.0 |

### Edge Cases

- **sqrt(x)** for x < 0 raises **ValueError**.

- **factorial(n)** requires n a non-negative integer; otherwise ValueError or OverflowError.

- For degree-to-radian conversion: **radians = degrees * math.pi / 180**.

In [ ]:
print(math.ceil(3.2), math.floor(3.8), math.trunc(-3.7))
print(math.factorial(5), math.hypot(3, 4), math.sqrt(2))

In [ ]:
# Degree to radian: use for sin/cos/tan
degrees = 90
rad = math.radians(degrees)
print("90 degrees in radians:", rad, "sin(90°) =", math.sin(rad))

In [ ]:
# Compare ceil, floor, trunc — note trunc "cuts toward zero" (different from floor for negatives)
print("ceil(3.2)=", math.ceil(3.2), "| floor(3.8)=", math.floor(3.8))
print("floor(-3.7)=", math.floor(-3.7), "| trunc(-3.7)=", math.trunc(-3.7), "| int(-3.7)=", int(-3.7))

In [ ]:
# Edge cases: sqrt of negative raises ValueError; handle with try/except
try:
    math.sqrt(-1)
except ValueError as e:
    print("math.sqrt(-1) raises ValueError:", e)
# factorial(-1) would raise ValueError; factorial(3.5) would raise ValueError (must be int)
try:
    math.factorial(-1)
except ValueError as e:
    print("math.factorial(-1) raises ValueError:", e)

---
## 4. The random Module (PCAP 1.3)

### Concepts

The **random** module implements pseudo-random number generation. For **reproducible** sequences (e.g. in tests or demos), call **random.seed(a)** with a fixed integer before generating.

### Function Reference (PCAP 1.3)

| Function | Description | Example |
|----------|-------------|--------|
| **random()** | Next float in [0.0, 1.0) | random.random() |
| **seed(a=None)** | Initialize RNG; same seed -> same sequence | seed(42) |
| **choice(seq)** | One random element from non-empty sequence | choice([1,2,3]) |
| **sample(seq, k)** | k unique elements (no replacement); k <= len(seq) | sample(range(10), 3) |

### Edge Cases

- **choice(seq)** requires non-empty seq; otherwise **IndexError**.

- **sample(seq, k)** requires k <= len(seq); otherwise **ValueError**.

- **seed()** is useful for debugging and unit tests; do not use for security-sensitive randomness (use **secrets** module instead).

In [ ]:
import random
random.seed(42)
print(random.random(), random.choice(["a","b","c"]), random.sample(range(10), 3))

In [ ]:
# Edge case: choice() on empty list raises IndexError
try:
    random.choice([])
except IndexError as e:
    print("random.choice([]) raises IndexError:", e)
# Edge case: sample(seq, k) requires k <= len(seq)
try:
    random.sample([1, 2, 3], 5)  # k=5 > len(seq)=3
except ValueError as e:
    print("random.sample([1,2,3], 5) raises ValueError:", e)

In [ ]:
# Reproducibility: same seed -> same sequence. Run this cell twice and compare.
random.seed(42)
first_three = [random.random() for _ in range(3)]
print("With seed(42), first three random():", first_three)
random.seed(42)
again = [random.random() for _ in range(3)]
print("Same seed again:", again)
print("Identical:", first_three == again)

---
## 5. The platform Module (PCAP 1.4)

### Concepts

The **platform** module reveals information about the **host** (OS, machine, Python implementation). Use it for environment reports, conditional logic (e.g. different paths on Windows vs Linux), or diagnostics.

### Function Reference (PCAP 1.4)

| Function | Description | Typical values |
|----------|-------------|----------------|
| **platform()** | Human-readable platform string | Windows-10-..., Linux-... |
| **machine()** | Machine type | AMD64, x86_64, arm64 |
| **processor()** | Processor name (may be empty) | Intel64, ... |
| **system()** | OS name | Windows, Linux, Darwin |
| **version()** | OS version string | 10.0.26200, ... |
| **python_implementation()** | Interpreter | CPython, PyPy |
| **python_version_tuple()** | (major, minor, patch) | ('3', '10', '0') |

In [ ]:
print(platform.system(), platform.machine(), platform.python_version_tuple())

In [ ]:
# More platform details: build a short environment report
print("OS:", platform.system())
print("Machine:", platform.machine())
print("Python:", ".".join(platform.python_version_tuple()))
print("Implementation:", platform.python_implementation())
print("Full platform string:", platform.platform())

---
## 6. Built-in and Related Functions

- **dir(obj)** – list of attribute names (module, class, or instance).

- **hasattr(obj, name)** – True if obj has attribute name.

- **getattr(obj, name [, default])** – get attribute; optional default if missing.

- **importlib.reload(module)** – reload a module (useful in REPL after editing).

Use **dir(module)** to discover what a module exports before reading the docs.

In [ ]:
# Discover and use attributes
print("math has 'sqrt':", hasattr(math, "sqrt"))
print("getattr(math, 'sqrt')(9) =", getattr(math, "sqrt")(9))

In [ ]:
# Example: hasattr — check if a module has an attribute before using it
print("math has 'sqrt':", hasattr(math, "sqrt"))
print("math has 'sqr' (typo):", hasattr(math, "sqr"))
# Safe pattern: check before calling
if hasattr(math, "sqrt"):
    print("Safe call:", math.sqrt(49))

---
## 7. Common Errors and Edge Cases

- **ImportError** / **ModuleNotFoundError**: module not found in **sys.path**; check spelling and that the file/package exists.

- **AttributeError**: you used a name the module does not have (e.g. `math.sqr` instead of `math.sqrt`). Use **dir(module)** to list names.

- **ValueError** from **math.sqrt(-1)** or **math.factorial(-1)**; validate inputs or use try/except.

- **from module import ***: pollutes the namespace and hides where names come from; avoid in production.

In [ ]:
# Example: importlib.reload — reload a module after editing (useful in REPL/notebooks)
import importlib
# If you edit a .py file and want to pick up changes without restarting:
# importlib.reload(some_module)
print("importlib.reload(module) re-executes the module and updates its namespace")

In [ ]:
# Example: ModuleNotFoundError — happens when module not in sys.path
try:
    import nonexistent_module_xyz
except ModuleNotFoundError as e:
    print("ModuleNotFoundError:", e)

In [ ]:
# Common error: AttributeError when the name does not exist
try:
    x = math.sqr(9)  # typo: should be sqrt
except AttributeError as e:
    print("AttributeError (math has no 'sqr'):", e)
# Correct name:
print("Correct call math.sqrt(9) =", math.sqrt(9))

In [ ]:
# getattr with default: safe access when attribute might be missing
fn = getattr(math, "sqrt", None)
print("getattr(math, 'sqrt', None):", fn)
print("Callable:", callable(fn))
if fn:
    print("fn(16) =", fn(16))
missing = getattr(math, "nonexistent", "default_value")
print("getattr(math, 'nonexistent', 'default_value'):", missing)

---
## 8. More Examples

**Example: Random angle and unit vector (math + random)**

In [ ]:
random.seed(0)
angle = random.random() * 2 * math.pi
x, y = math.cos(angle), math.sin(angle)
print("Random angle (rad):", round(angle, 4), "-> unit vector (x,y):", (round(x, 4), round(y, 4)))

**Example: OS-specific greeting (platform)**

In [ ]:
sys_name = platform.system()
if sys_name == "Windows":
    print("Running on Windows")
elif sys_name == "Linux":
    print("Running on Linux")
else:
    print("Running on", sys_name)
print("Machine:", platform.machine(), "| Python:", ".".join(platform.python_version_tuple()))

**Example: List names in math that start with 'f'; use hypot for distance**

In [ ]:
print("math names starting with 'f':", [n for n in dir(math) if n.startswith("f")])
print("Distance (0,0) to (1,1):", math.hypot(1, 1))

---
## 9. Practice

**Practice 1:** List all names in **math** that start with `'f'`. (You can use a list comprehension and `str.startswith`.)

In [ ]:
print([n for n in dir(math) if n.startswith("f")])

**Practice 2:** Use **math.hypot** to compute the distance from the origin (0, 0) to the point (3, 4). Verify it equals 5.0.

In [ ]:
print(math.hypot(3, 4))

**Practice 3:** Use **random.seed(42)** then **random.choice** twice on the list `["red", "green", "blue"]`. Run the cell twice: you should get the same two values each time.

In [ ]:
random.seed(42)
colors = ["red", "green", "blue"]
print(random.choice(colors), random.choice(colors))

**Practice 4:** Write a one-liner that uses **math.floor** and **math.ceil** to round a float `x` to the nearest integer (0.5 rounds up). Test with 3.2 and 3.7.

In [ ]:
def round_half_up(x):
    return math.ceil(x) if x - math.floor(x) >= 0.5 else math.floor(x)
print(round_half_up(3.2), round_half_up(3.7))

**Practice 5:** Build a short "environment report" string using **platform**: system name, machine, and Python version (e.g. from **python_version_tuple()**). Print it.

In [ ]:
report = f"OS: {platform.system()}, Machine: {platform.machine()}, Python: {'.'.join(platform.python_version_tuple())}"
print(report)

**Practice 6:** Append a directory path string to **sys.path** (e.g. `"."`), then use **dir()** on the **math** module and print its first 5 names. (Do not remove the appended path from sys.path for this exercise.)

In [ ]:
sys.path.append(".")
print("First 5 names in math:", dir(math)[:5])